# Python 动态特性详解 —— 教学级 Notebook

> **业务场景假设**：目标目录位于 `LLMOps/llmops-api`，本文所有工程示例统一取材于
> **「LLMOps API 服务开发」**——模型网关、插件化 Provider、动态配置、API 客户端封装等场景。
> （原始需求中的 `{{业务场景描述}}` 占位符未填写，按此路径推断，如需更换场景可告知我重新生成。）

## 学习目标

1. 理解「动态」的本质：类型、属性、方法、类、模块都可以在**运行时**创建、修改、查询
2. 掌握 9 大动态机制：动态类型 / 动态属性 / 属性访问协议 / 动态类 / 反射 / 猴子补丁 / 动态导入 / 动态执行 / 动态协议
3. 能读懂主流框架（ORM、插件系统、mock、路由）背后的动态特性实现原理
4. 建立「动态 vs 安全 / 可维护性」的工程权衡意识，避开 10 个高频陷阱

## 一句话本质

> **Python 的动态特性让你能在运行时"改写规则"——它既是框架和库的基石，也是 bug 和安全问题的来源。用得好是魔法，用不好是灾难。**

> **本章解决什么问题**：动态特性建立在 Python 对象模型的几个基础事实上。
把这 4 件事想清楚，后面所有"魔法"都只是它们的组合。

## 0.1 一切皆对象

Python 中数字、字符串、函数、类本身，全都是对象——都可以赋值给变量、作为参数传递、动态查询类型。

In [1]:
# 【0.1】验证：函数、类也是对象，类型可运行时查询
for v in [10, "hello", [1, 2], print, int]:   # 依次取：整数 / 字符串 / 列表 / 内置函数 / 类本身
    print(f"type = {type(v)!s:<32} value = {v!r}")
# 输出：int / str / list / builtin_function_or_method / type —— 全是对象

type = <class 'int'>                    value = 10
type = <class 'str'>                    value = 'hello'
type = <class 'list'>                   value = [1, 2]
type = <class 'builtin_function_or_method'> value = <built-in function print>
type = <class 'type'>                   value = <class 'int'>


## 0.2 变量是「名字标签」，不是盒子

变量没有类型，只有对象有类型。赋值只是把一个名字**绑定**到对象上，运行时可以重新绑定到任意其它类型的对象。

In [2]:
# 【0.2】同一个名字先后绑定到三种不同类型的对象 —— 合法且常见
x = 10
print(type(x))     # <class 'int'>

x = "hello"
print(type(x))     # <class 'str'>

x = [1, 2, 3]
print(type(x))     # <class 'list'>

<class 'int'>
<class 'str'>
<class 'list'>


## 0.3 命名空间 = 字典

「名字 → 对象」的映射就存在字典里：模块用 `globals()`，函数用 `locals()`，实例用 `__dict__`，类也有自己的命名空间。
**所以"动态"的物理本质是：这些字典可以在运行时增删改查。**

In [3]:
# 【0.3】验证：各种命名空间本质上是字典
a = 100                       # 模块（全局）命名空间

def demo_ns():
    b = 1                     # 函数局部命名空间
    return locals()

print("函数局部命名空间:", demo_ns())          # {'b': 1}
print("a 在全局命名空间:", "a" in globals())    # True

class C:
    v = 1                     # 类命名空间

print("类命名空间:", vars(C))                   # 含 '__module__'、'__qualname__'、'v' 等

函数局部命名空间: {'b': 1}
a 在全局命名空间: True
类命名空间: {'__module__': '__main__', '__firstlineno__': 11, 'v': 1, '__static_attributes__': (), '__dict__': <attribute '__dict__' of 'C' objects>, '__weakref__': <attribute '__weakref__' of 'C' objects>, '__doc__': None}


## 0.4 属性查找顺序（贯穿全文的核心规则）

访问 `obj.name` 时，查找顺序为：

1. **实例 `__dict__`**（数据描述器优先级更高，见第 13.2 节）
2. **类及基类的 `__dict__`**（按 MRO 顺序）
3. 都找不到 → 调用 `__getattr__`（若定义）
4. 仍没有 → 抛 `AttributeError`

而 `__getattribute__` 则拦截**所有**属性访问（第 3 章详解）。

In [4]:
# 【0.4】验证属性查找顺序：实例 __dict__ → 类 __dict__ → 基类 __dict__
class A:
    pass

class B(A):
    pass

b = B()
B.attr = "来自类 B"        # 动态写入类命名空间
A.attr = "来自类 A"        # 动态写入基类命名空间

print("1) 实例无、B 有:", b.attr)     # 来自类 B（先查类 B 再查基类 A）
del B.attr
print("2) 删除 B 的后:", b.attr)      # 来自类 A（沿 MRO 继续向上找）
b.attr = "实例自己"                   # 动态写入实例 __dict__ —— 优先级最高
print("3) 实例有之后:", b.attr)       # 实例自己

1) 实例无、B 有: 来自类 B
2) 删除 B 的后: 来自类 A
3) 实例有之后: 实例自己


## 0.5 可调用对象

任何实现了 `__call__` 的对象都像函数一样可调用——这是「对象行为可在运行时赋予」的直接体现。

In [5]:
# 【0.5】验证：实例也可以是"函数"
class Greeter:
    def __call__(self, name):        # 实现 __call__ → 实例可调用
        return f"你好, {name}"

g = Greeter()
print("g 可调用:", callable(g))       # True
print("42 可调用:", callable(42))     # False
print(g("动态特性"))                  # 你好, 动态特性

g 可调用: True
42 可调用: False
你好, 动态特性


# 第 1 章 动态类型

> **本章解决什么问题**：类型检查被推迟到运行时，这带来了哪些能力与代价。

核心结论：**变量没有类型，只有对象有类型；类型检查发生在运行时。**

| 维度 | 动态类型 | 静态类型 |
|---|---|---|
| 类型检查 | 运行时 | 编译时 |
| 灵活性 | 高 | 低 |
| 安全性 | 低 | 高 |
| 性能 | 略低 | 略高 |
| 代码量 | 少 | 多 |

> Python 3.5+ 的类型注解是「**可选静态类型**」：仅供 IDE / mypy 检查，**运行时不强制**（1.3 节验证）。

In [6]:
# 【1.0】同一个名字绑定到不同类型的对象，运行时随时可查类型
x = 10
print(type(x))        # <class 'int'>

x = "hello"
print(type(x))        # <class 'str'>

x = [1, 2, 3]
print(type(x))        # <class 'list'>

<class 'int'>
<class 'str'>
<class 'list'>


## 1.1 类型检查在运行时

`a + b` 是否合法，取决于**运行时** a、b 的实际类型是否支持 `+`。同一个函数天然泛化到所有支持该操作的对象。

In [7]:
# 【1.1】验证：同一个 add 函数对 int / str / list 都有效
def add(a, b):
    return a + b

print(add(1, 2))        # 3       —— 数值加法
print(add("a", "b"))    # ab      —— 字符串拼接
print(add([1], [2]))    # [1, 2]  —— 列表拼接

# 类型不兼容时，报错发生在【运行时调用处】，而非函数定义处：
try:
    add(1, "a")
except TypeError as e:
    print("TypeError:", e)   # TypeError: unsupported operand type(s) for +: 'int' and 'str'

3
ab
[1, 2]
TypeError: unsupported operand type(s) for +: 'int' and 'str'


## 1.2 鸭子类型

> "如果它走起来像鸭子，叫起来像鸭子，那它就是鸭子。"

不关心对象**是什么类型**，只关心它**有什么行为**。这是 Python 动态协议的哲学基础。

In [8]:
# 【1.2】鸭子类型：make_quack 只关心对象会不会 quack
class Duck:
    def quack(self):
        print("嘎嘎")

class Person:
    def quack(self):
        print("学鸭子叫")

def make_quack(obj):     # 不检查类型，直接调用行为
    obj.quack()

make_quack(Duck())       # 嘎嘎
make_quack(Person())     # 学鸭子叫

嘎嘎
学鸭子叫


## 1.3 类型注解：运行时不强制（验证）

> ⚠️ **修正说明**：源文档此处仅给出结论。这里补充**可运行的验证代码**：
> 注解只写入 `__annotations__`，解释器不据此做任何拦截。

In [9]:
# 【1.3】验证：类型注解在运行时完全不强制
def double(x: int) -> int:      # 声明 x 是 int
    return x * 2

print(double("ha"))                     # haha —— 传 str 也照常执行！
print("注解内容:", double.__annotations__)  # {'x': <class 'int'>, 'return': <class 'int'>}
# 注解只是"元数据"，供 mypy / IDE 使用；运行时零拦截

haha
注解内容: {'x': <class 'int'>, 'return': <class 'int'>}


# 第 2 章 动态属性

> **本章解决什么问题**：属性不必在类定义时声明，可以在运行时添加、修改、删除——这是配置对象、数据对象的技术底座。

要点速览：

- 实例属性直接存放在实例 `__dict__`，随时增删
- `setattr / getattr / hasattr / delattr`：用**字符串**操作属性（反射式 API）
- `types.MethodType`：把普通函数动态绑定为方法

In [10]:
# 【2.1】运行时添加 / 删除属性
class User:
    pass                      # 空类，什么都没声明

u = User()
u.name = "Alice"              # 运行时添加属性
u.age = 30
print(u.__dict__)             # {'name': 'Alice', 'age': 30}

del u.age                     # 运行时删除属性
print(u.__dict__)             # {'name': 'Alice'}

{'name': 'Alice', 'age': 30}
{'name': 'Alice'}


In [11]:
# 【2.2】动态方法：types.MethodType 把函数绑定为【实例】方法
import types

class User:
    pass

def greet(self):
    return f"Hello, {self.name}"

u = User()
u.name = "Alice"
u.greet = types.MethodType(greet, u)   # 绑定到单个实例：只有 u 拥有
print(u.greet())                       # Hello, Alice

# 【2.3】也可以直接绑到【类】上：所有实例共享
User.greet = greet
u2 = User()
u2.name = "Bob"
print(u2.greet())                      # Hello, Bob

Hello, Alice
Hello, Bob


## 2.4 setattr / getattr / hasattr / delattr

| 函数 | 作用 | 等价写法 |
|---|---|---|
| `setattr(obj, "k", v)` | 设置属性 | `obj.k = v` |
| `getattr(obj, "k")` | 读取属性 | `obj.k` |
| `getattr(obj, "k", 默认)` | 读取，找不到返回默认 | ——（属性访问做不到） |
| `hasattr(obj, "k")` | 判断是否存在 | —— |
| `delattr(obj, "k")` | 删除属性 | `del obj.k` |

In [12]:
# 【2.4】setattr / getattr / hasattr / delattr 基本用法
class Config:
    pass

c = Config()

setattr(c, "debug", True)               # 等价 c.debug = True
print(getattr(c, "debug"))              # True
print(getattr(c, "missing", "默认"))     # 默认（找不到时给默认值，不抛异常）
print(hasattr(c, "debug"))              # True
delattr(c, "debug")                     # 等价 del c.debug
print(hasattr(c, "debug"))              # False

True
默认
True
False


In [13]:
# 【2.5】批量设置：配置字典 → 对象属性（LLMOps 场景：把 YAML 配置挂到对象上）
settings = {"host": "localhost", "port": 8080, "debug": False}

class RuntimeConfig:
    pass

rc = RuntimeConfig()
for k, v in settings.items():      # 键是字符串 → 只能用 setattr
    setattr(rc, k, v)

print(rc.host, rc.port, rc.debug)  # localhost 8080 False
print(vars(rc))                    # {'host': 'localhost', 'port': 8080, 'debug': False}

localhost 8080 False
{'host': 'localhost', 'port': 8080, 'debug': False}


## 2.6 动态属性访问：`__getattr__` 实现懒加载

`__getattr__` 只在**常规查找失败后**才被调用。利用这一点可以做「首次访问才计算并缓存」。

In [14]:
# 【2.6】Lazy：属性首次访问时生成，写入 __dict__ 后不再触发 __getattr__
class Lazy:
    def __getattr__(self, name):
        print(f"  [懒加载] 首次访问 {name}，生成并缓存")
        value = name.upper()
        setattr(self, name, value)   # 写入实例 __dict__ → 下次直接命中，不再进 __getattr__
        return value

l = Lazy()
print(l.foo)            # 触发 __getattr__：打印懒加载日志，返回 FOO
print(l.foo)            # 直接命中 __dict__：无日志，返回 FOO
print("实例缓存:", l.__dict__)   # {'foo': 'FOO'}

  [懒加载] 首次访问 foo，生成并缓存
FOO
FOO
实例缓存: {'foo': 'FOO'}


# 第 3 章 属性访问协议（四个拦截钩子）

> **本章解决什么问题**：想在属性读写删的瞬间插入自定义逻辑（代理、校验、延迟加载），就需要这四个魔术方法。

| 钩子 | 触发时机 | 典型用途 | 风险 |
|---|---|---|---|
| `__getattr__` | 常规查找**失败后** | 代理、懒加载、默认值 | 拼写错误被吞掉 |
| `__getattribute__` | **所有**属性访问 | 审计、拦截 | 极易死循环，慎用 |
| `__setattr__` | 所有赋值 | 校验、类型检查 | 忘调 super() 会丢属性 |
| `__delattr__` | 所有删除 | 审计、清理 | 同上 |

In [15]:
# 【3.1】__getattr__：属性找不到时调用（提供默认值）
class Demo:
    def __getattr__(self, name):
        return f"默认值: {name}"     # 任何未定义属性都返回默认值

d = Demo()
print(d.anything)    # 默认值: anything

默认值: anything


In [16]:
# 【3.2】__getattribute__：拦截【所有】属性访问（注意必须转发给 super()）
class Audited:
    def __getattribute__(self, name):
        print(f"  [审计] 访问 {name}")
        return super().__getattribute__(name)   # 关键：交给默认机制，否则死循环

a = Audited()
a.x = 1
print(a.x)
# 输出：
#   [审计] 访问 x
#   1

  [审计] 访问 x
1


In [17]:
# 【3.3】__setattr__ / __delattr__：拦截赋值与删除
class Traced:
    def __setattr__(self, name, value):
        print(f"  [拦截] 设置 {name} = {value!r}")
        super().__setattr__(name, value)        # 关键：真正落盘
    def __delattr__(self, name):
        print(f"  [拦截] 删除 {name}")
        super().__delattr__(name)

t = Traced()
t.x = 1                     #   [拦截] 设置 x = 1
print(t.x)                  # 1
del t.x                     #   [拦截] 删除 x
print("删除后还存在:", hasattr(t, "x"))   # False

  [拦截] 设置 x = 1
1
  [拦截] 删除 x
删除后还存在: False


## 3.5 使用场景：代理对象

`__getattr__` + `getattr` 组合是最经典的**代理模式**：把不认识的属性全部转发给内部对象。

适用场景：代理对象 / 延迟加载 / 配置对象 / ORM 字段 / API 包装。

In [18]:
# 【3.5】APIWrapper：把属性访问转发给被包装的 client
class APIWrapper:
    def __init__(self, client):
        self._client = client            # 真实对象

    def __getattr__(self, name):         # 只拦截"找不到"的属性
        return getattr(self._client, name)   # 转发给内部对象

w = APIWrapper([10, 20])                 # 用 list 模拟 client
w.append(30)                             # list.append 被透传执行
print("内部对象:", w._client)             # [10, 20, 30]

内部对象: [10, 20, 30]


# 第 4 章 动态创建类

> **本章解决什么问题**：类本身也是对象，可以在运行时"生产"类——ORM 模型、插件系统都靠它。

`class` 语句只是 `type(name, bases, namespace)` 的语法糖：

```python
User = type("User", (Base,), {"name": "Alice", "greet": func})
#            │        │          └── namespace：类字典（属性/方法）
#            │        └── bases：基类元组
#            └── name：类名
```

In [19]:
# 【4.1】type() 三参数形式：动态创建一个完整的类
User = type("User", (), {"name": "Alice", "greet": lambda self: f"Hi, {self.name}"})
u = User()
print(u.name)      # Alice
print(u.greet())   # Hi, Alice
print(type(User))  # <class 'type'> —— 类的 type 就是元类 type

Alice
Hi, Alice
<class 'type'>


In [20]:
# 【4.2】动态添加方法（等价写法：先建类，再塞方法）
def greet(self):
    return f"Hello, {self.name}"

User2 = type("User2", (), {
    "name": "Alice",
    "greet": greet,          # 函数放进 namespace 即成为方法
})

print(User2().greet())       # Hello, Alice

Hello, Alice


In [21]:
# 【4.3】动态继承：bases 里放上基类即可
class Base:
    def hello(self):
        return "hello"

Dynamic = type("Dynamic", (Base,), {"world": lambda self: "world"})
d = Dynamic()
print(d.hello())   # hello —— 继承自 Base
print(d.world())   # world —— 自己的

hello
world


## 4.4 元类 vs `__init_subclass__` vs 类装饰器

三种「定制类创建」的手段，**优先级从高到低**（能简单勿复杂）：

| 手段 | 拦截对象 | 复杂度 | 推荐度 |
|---|---|---|---|
| 类装饰器 | 类创建后加工 | 低 | ★★★ |
| `__init_subclass__` | 基类被继承时 | 低 | ★★★ |
| 元类 | 类创建全过程 | 高 | ★（框架级才用） |

In [22]:
# 【4.4】元类：拦截"类的创建"本身
class Meta(type):
    def __new__(mcls, name, bases, namespace):
        namespace["created"] = True        # 在类字典里注入属性
        return super().__new__(mcls, name, bases, namespace)

class WithMeta(metaclass=Meta):
    pass

print(WithMeta.created)    # True —— 类一出生就带 created 属性

True


In [23]:
# 【4.5】__init_subclass__：子类被定义时自动回调（注册表模式的核心）
class Base:
    registry = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Base.registry[cls.__name__] = cls    # 每定义一个子类，自动登记

class User(Base): pass
class Order(Base): pass

print(Base.registry)
# {'User': <class '__main__.User'>, 'Order': <class '__main__.Order'>}

{'User': <class '__main__.User'>, 'Order': <class '__main__.Order'>}


In [24]:
# 【4.6】类装饰器：给类"后加工"（比元类简单得多）
def add_repr(cls):
    def __repr__(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class Account:
    def __init__(self, name):
        self.name = name

print(Account("Alice"))    # Account(name='Alice')

Account(name='Alice')


## 4.7 使用场景

ORM 模型（字段→类属性）、插件系统（注册类）、配置类、API 客户端、测试 mock——第 13、15 章将逐一落地。

# 第 5 章 反射与内省

> **本章解决什么问题**：程序在运行时"看自己"——查询对象有什么属性方法、从哪来、签名是什么。

| 工具 | 用途 |
|---|---|
| `dir()` | 列出对象全部属性名 |
| `type / isinstance / issubclass` | 类型判断 |
| `vars()` | 取命名空间字典 |
| `id / is / hash` | 对象身份 |
| `inspect` 模块 | 签名、源码、调用栈 |

In [25]:
# 【5.1】dir / isinstance / issubclass
print(type(1))                     # <class 'int'>
print(isinstance(1, int))          # True
print(issubclass(bool, int))       # True —— bool 是 int 的子类
print(dir("hello")[:6], "...")     # 字符串对象可用属性（截取前 6 个展示）

<class 'int'>
True
True
['__add__', '__class__', '__contains__', '__delattr__', '__dir__', '__doc__'] ...


In [26]:
# 【5.2】vars()：取对象的命名空间字典
class UserV:
    def __init__(self):
        self.name = "Alice"

u = UserV()
print("实例 __dict__:", vars(u))            # {'name': 'Alice'}
print("类命名空间键:", sorted(vars(UserV)))  # ['__dict__', '__init__', '__module__', '__qualname__', '__weakref__']

实例 __dict__: {'name': 'Alice'}
类命名空间键: ['__dict__', '__doc__', '__firstlineno__', '__init__', '__module__', '__static_attributes__', '__weakref__']


> ⚠️ **修正说明（确定性改造）**：源文档示例直接 `print(id(x))`、`print(hash("hello"))`。
> `id` 返回内存地址、字符串 `hash` 受 `PYTHONHASHSEED` 随机化影响，**每次运行输出都不同**，
> 违反"输出可复现"要求。这里改造成**结论确定**的等价验证。

In [27]:
# 【5.3】对象身份：用结论而非原始数值验证（保证输出可复现）
x = [1, 2, 3]
y = x            # 同一个对象的另一个名字
z = [1, 2, 3]    # 值相等的新对象

print("x is y:", x is y)                  # True  —— is 比较对象身份
print("x is z:", x is z)                  # False —— 值相等但不是同一个对象
print("x == z:", x == z)                  # True  —— == 比较值
print("id(x) == id(y):", id(x) == id(y))  # True  —— 同一对象 id 必然相同

# hash：整数哈希恒等于自身（输出确定）
print("hash(42) == 42:", hash(42) == 42)             # True
# 字符串哈希受 PYTHONHASHSEED 影响，跨进程不确定；但同一进程内稳定：
print("同进程内字符串哈希稳定:", hash("hello") == hash("hello"))   # True

x is y: True
x is z: False
x == z: True
id(x) == id(y): True
hash(42) == 42: True
同进程内字符串哈希稳定: True


In [28]:
# 【5.4】inspect：函数签名与源码（框架做依赖注入 / 文档生成的依据）
import inspect

def greet(name: str, age: int = 0) -> str:
    return f"{name}, {age}"

print("签名:", inspect.signature(greet))    # (name: str, age: int = 0) -> str
print("是函数:", inspect.isfunction(greet))  # True
print("源码:")
print(inspect.getsource(greet))             # IPython 会缓存 cell 源码，可正常取出

签名: (name: str, age: int = 0) -> str
是函数: True
源码:
def greet(name: str, age: int = 0) -> str:
    return f"{name}, {age}"



In [29]:
# 【5.5】运行时获取调用者信息（日志 / 调试场景）
import inspect

def current_function_name():
    # currentframe() 是当前帧，f_back 是调用者的帧，co_name 是函数名
    return inspect.currentframe().f_back.f_code.co_name

def caller():
    return current_function_name()

print("当前函数名:", caller())    # caller —— 运行时拿到了调用链信息

当前函数名: caller


# 第 6 章 猴子补丁（Monkey Patch）

> **本章解决什么问题**：运行时替换已有的类 / 模块 / 实例的属性——测试打桩、紧急修 bug 的常用手段，也是最危险的动态特性。

| 对象 | 做法 | 影响范围 |
|---|---|---|
| 类 | `User.greet = new_greet` | 该类**所有**实例 |
| 模块 | `math.sqrt = logged_sqrt` | **全局**所有使用方 |
| 实例 | `u.greet = ...` | 仅该实例 |

In [30]:
# 【6.1】修改类：替换方法，全局生效
class User:
    def greet(self):
        return "hello"

def new_greet(self):
    return "HELLO"

print("打补丁前:", User().greet())   # hello
User.greet = new_greet               # 猴子补丁：替换类方法
print("打补丁后:", User().greet())   # HELLO

打补丁前: hello
打补丁后: HELLO


In [31]:
# 【6.2】修改模块：给 math.sqrt 加日志（演示后立即恢复，避免污染后续章节）
import math

original_sqrt = math.sqrt

def logged_sqrt(x):
    print(f"  [日志] sqrt({x})")       # 前置逻辑：性能监控 / 审计
    return original_sqrt(x)            # 调用原函数

math.sqrt = logged_sqrt
print("补丁生效中:", math.sqrt(4))     # 打印日志并返回 4.0

math.sqrt = original_sqrt              # 【重要】用完恢复
print("已恢复:", math.sqrt(4))         # 4.0（无日志）

  [日志] sqrt(4)
补丁生效中: 2.0
已恢复: 2.0


In [32]:
# 【6.3】修改实例：只影响单个对象
class Greeter:
    def greet(self):
        return "class version"

g1, g2 = Greeter(), Greeter()
g1.greet = lambda: "instance version"   # 实例属性遮蔽类方法（注意：无 self）
print(g1.greet())    # instance version —— 只影响 g1
print(g2.greet())    # class version    —— g2 不受影响

instance version
class version


## 6.4 测试中的 mock：受控的猴子补丁

`unittest.mock.patch` 是**自动打补丁 + 自动恢复**的正规军做法，测试结束自动还原，避免 6.2 的手动恢复风险。

In [33]:
# 【6.4】unittest.mock.patch：进出上下文管理器自动打/撕补丁
from unittest.mock import patch
import math

with patch("math.floor", return_value=42):   # 作用域内：math.floor 恒返回 42
    print("补丁内:", math.floor(3.7))        # 42
print("补丁外:", math.floor(3.7))            # 3 —— 已自动恢复

补丁内: 42
补丁外: 3


## 6.5 使用场景与风险

**适用**：测试打桩 / 修复第三方库 bug / 性能监控 / 功能扩展

| 风险 | 说明 |
|---|---|
| 难以调试 | 阅读 `User().greet()` 的代码看不出方法被换过 |
| 影响全局 | 模块级补丁波及所有调用方 |
| 版本易失效 | 上游一升级，补丁目标可能不存在 |
| 破坏封装 | 绕过类的公开接口 |

**纪律**：猴子补丁只用于测试或紧急修复；生产代码优先显式实现。

# 第 7 章 动态导入

> **本章解决什么问题**：import 什么不再写死在文件头，而是运行时按字符串决定——插件架构的核心。

| API | 说明 |
|---|---|
| `importlib.import_module("a.b")` | 按字符串导入模块（推荐） |
| `importlib.reload(module)` | 重新加载已导入的模块（热重载） |
| `__import__("a.b", fromlist=[...])` | 底层内置函数（不建议直接用） |
| `pkgutil.iter_modules(path)` | 枚举包下的子模块（插件发现） |

In [34]:
# 【7.1】importlib.import_module：字符串导入
import importlib

module = importlib.import_module("os.path")
print(module.join("a", "b"))     # a/b
# 等价于：from os import path

a/b


## 7.2 准备一个真实的插件包

为了「真实可运行」，下面在临时目录里创建一个 `plugins` 包（含 `pdf`、`csv_export` 两个插件和 1 个非插件模块 `utils`），后续 7.2 / 7.5 / 7.6 节都基于它。

> ⚠️ **修正说明（可复现性）**：`pkgutil.iter_modules` 的返回顺序在不同平台不保证有序，
> 后续枚举代码统一加 `sorted()` 保证每次执行输出一致。

In [35]:
# 【7.2-setup】创建真实的插件包（写入系统临时目录，不污染学习目录）
import os, sys, tempfile

_plug_root = tempfile.mkdtemp(prefix="nb_plugins_")
_pkg = os.path.join(_plug_root, "plugins")
os.makedirs(_pkg)
open(os.path.join(_pkg, "__init__.py"), "w").close()      # 包标识

def _write(name, text):
    with open(os.path.join(_pkg, name), "w") as f:
        f.write(text)

_write("pdf.py", "def register():\n    return {'name': 'pdf', 'type': 'exporter'}\n")
_write("csv_export.py", "def register():\n    return {'name': 'csv', 'type': 'exporter'}\n")
_write("utils.py", "VERSION = '1.0'\n")                  # 没有 register —— 应被过滤

sys.path.insert(0, _plug_root)
print("插件包已就绪，包含模块:", sorted(m for m in os.listdir(_pkg) if m.endswith(".py")))
# ['__init__.py', 'csv_export.py', 'pdf.py', 'utils.py']

插件包已就绪，包含模块:

 ['__init__.py', 'csv_export.py', 'pdf.py', 'utils.py']


In [36]:
# 【7.2】按字符串加载指定插件（LLMOps 场景：按用户请求加载导出器）
import importlib

def load_plugin(name):
    module = importlib.import_module(f"plugins.{name}")   # 字符串拼接模块路径
    if hasattr(module, "register"):
        return module.register()
    raise AttributeError(f"plugins.{name} 不是合法插件（缺少 register）")

print(load_plugin("pdf"))
print(load_plugin("csv_export"))

{'name': 'pdf', 'type': 'exporter'}
{'name': 'csv', 'type': 'exporter'}


In [37]:
# 【7.3】reload：模块已导入后再改文件，必须 reload 才能看到新版本
import importlib, os

_mod_path = os.path.join(_pkg, "counter.py")

def write_counter(version):
    with open(_mod_path, "w") as f:
        f.write(f"VERSION = {version}\n")

write_counter(1)
import plugins.counter as counter
print("首次导入:", counter.VERSION)     # 1

write_counter(2)                        # 模拟磁盘上的模块被修改
print("未 reload  :", counter.VERSION)  # 1 —— 已导入的模块对象不会自动刷新
importlib.reload(counter)
print("reload 后  :", counter.VERSION)  # 2 —— 强制重新执行模块代码

首次导入: 1


未 reload  : 1


reload 后  : 2


In [38]:
# 【7.4】__import__：import 语句背后真正调用的内置函数（了解即可，优先用 importlib）
mod = __import__("os.path", fromlist=["join"])
print(mod.join("a", "b"))    # a/b

a/b


In [39]:
# 【7.5】插件发现：pkgutil 枚举包内全部子模块（sorted 保证顺序可复现）
import importlib, pkgutil

def load_plugins(package):
    plugins = {}
    pkg = importlib.import_module(package)
    for _, name, _ in sorted(pkgutil.iter_modules(pkg.__path__)):   # 关键：sorted
        module = importlib.import_module(f"{package}.{name}")
        if hasattr(module, "register"):     # 约定优于配置：有 register 才算插件
            plugins[name] = module.register()
    return plugins

print(load_plugins("plugins"))
# {'csv_export': {'name': 'csv', 'type': 'exporter'}, 'pdf': {'name': 'pdf', 'type': 'exporter'}}
# 注意 utils 因无 register 被自动过滤

{'csv_export': {'name': 'csv', 'type': 'exporter'}, 'pdf': {'name': 'pdf', 'type': 'exporter'}}


## 7.6 使用场景

插件架构 / 延迟加载（用到才 import，加快启动）/ 配置驱动（模块名写在配置里）/ 动态路由 / 测试中替换实现。
第 15 章综合实战会把 7.5 的发现机制整合进 LLM 网关。

# 第 8 章 动态执行（eval / exec / compile）

> **本章解决什么问题**：把**字符串当代码**执行——模板引擎、配置脚本、交互式环境的底层机制，也是最大安全风险来源。

| 函数 | 输入 | 用途 |
|---|---|---|
| `eval(expr)` | 表达式 | 求值并返回结果 |
| `exec(stmts)` | 语句块 | 执行（无返回值） |
| `compile(src, file, mode)` | 源码 | 预编译为 code 对象 |

In [40]:
# 【8.1】eval：计算表达式
result = eval("1 + 2")
print(result)          # 3

x = 10
print(eval("x * 2"))   # 20 —— eval 默认在当前全局命名空间求值

3
20


In [41]:
# 【8.2】exec：执行语句（可以定义函数、类）
exec("y = 1 + 2")
print(y)               # 3 —— 语句效果落到了当前命名空间

code_str = '''
def dynamic_greet(name):
    return f"Hello, {name}"
'''
exec(code_str)                   # 字符串里定义的函数进入命名空间
print(dynamic_greet("Alice"))    # Hello, Alice

3
Hello, Alice


In [42]:
# 【8.3】compile：先编译后执行（同一份代码执行多次时只需编译一次）
code_obj = compile("z = 1 + 2", "<string>", "exec")
exec(code_obj)
print(z)               # 3

3


In [43]:
# 【8.4】作用域：给 exec 指定独立的 globals 字典（隔离污染）
env = {}
exec("w = 10", env)          # 写入 env，而不是当前命名空间
print(env["w"])              # 10
print("'w' 在当前命名空间:", "w" in globals())   # False —— 未污染

10
'w' 在当前命名空间: True


## 8.5 安全风险：为什么永远不要对不可信输入用 eval / exec

`eval` 能执行**任意表达式**——包括 `__import__`，等价于开放任意代码执行权限。

In [44]:
# 【8.5】攻击演示：一行 eval 就能"越狱"导入模块执行代码
attack = eval("__import__('math').sqrt(16)")   # 表达式里偷偷 import 了 math
print(attack)    # 4.0 —— 攻击者同样可以执行删除文件、发起网络请求等任意代码

# 危险写法（永远不要这样写）：
# eval(input("请输入表达式: "))

4.0


> ⚠️ **修正说明**：源文档的 `safe_eval` 使用 `ast.Num` 判断数字节点。
> `ast.Num` 自 Python 3.8 起废弃（3.12+ 触发 DeprecationWarning），**正确写法是 `ast.Constant`**。
> 下方代码已修正，并附拦截验证。另外，简单场景优先用标准库 `ast.literal_eval`。

In [45]:
# 【8.6】安全替代：AST 白名单校验后再求值（已修正 ast.Num → ast.Constant）
import ast

ALLOWED_NODES = (ast.Expression, ast.BinOp, ast.Constant,
                 ast.Add, ast.Sub, ast.Mult, ast.Div)   # 只允许：数字 + 四则运算

def safe_eval(expr):
    tree = ast.parse(expr, mode="eval")      # 先解析成语法树
    for node in ast.walk(tree):              # 遍历每个节点
        if not isinstance(node, ALLOWED_NODES):
            raise ValueError(f"不支持的表达式节点: {type(node).__name__}")
    return eval(compile(tree, "<safe>", "eval"))   # 校验通过才求值

print(safe_eval("1 + 2 * 3"))      # 7
print(safe_eval("(10 - 4) / 2"))   # 3.0
try:
    safe_eval("__import__('math').sqrt(16)")   # Call 节点不在白名单
except ValueError as e:
    print("已拦截:", e)    # 已拦截: 不支持的表达式节点: Call

7
3.0
已拦截: 不支持的表达式节点: Call


In [46]:
# 【8.6b】更简单的场景：ast.literal_eval 只解析字面量（数字/字符串/列表/字典）
import ast
print(ast.literal_eval("{'a': [1, 2], 'b': 3}"))   # {'a': [1, 2], 'b': 3}
try:
    ast.literal_eval("__import__('math')")          # 非字面量 → 直接拒绝
except ValueError as e:
    print("已拦截:", type(e).__name__)               # ValueError

{'a': [1, 2], 'b': 3}
已拦截: ValueError


# 第 9 章 动态协议与鸭子类型

> **本章解决什么问题**：如何给"鸭子类型"补上**可检查、可文档化**的契约——魔术方法协议与 `typing.Protocol`。

要点：

1. 鸭子类型：只看行为不看血缘
2. 魔术方法协议：实现 `__len__` 就支持 `len()`，实现 `__getitem__` 就支持下标——**协议是隐式继承的**
3. `typing.Protocol`：Python 3.8+ 的**结构化类型**，静态检查器可验证，`@runtime_checkable` 后还可 `isinstance`

In [47]:
# 【9.1】鸭子类型：process 不关心 A/B 是什么，只要求会 run()
def process(obj):
    obj.run()

class TaskA:
    def run(self):
        print("TaskA.run")

class TaskB:
    def run(self):
        print("TaskB.run")

process(TaskA())   # TaskA.run
process(TaskB())   # TaskB.run

TaskA.run
TaskB.run


In [48]:
# 【9.2】魔术方法协议：实现协议方法，内置函数自动支持你的类
class MyList:
    def __init__(self):
        self.data = []
    def __len__(self):            # 协议：支持 len()
        return len(self.data)
    def __getitem__(self, i):     # 协议：支持下标访问
        return self.data[i]
    def __iter__(self):           # 协议：支持 for 迭代
        return iter(self.data)

ml = MyList()
ml.data = [1, 2, 3]
print(len(ml))      # 3
print(ml[0])        # 1
print(list(ml))     # [1, 2, 3]

3
1
[1, 2, 3]


In [49]:
# 【9.3】typing.Protocol：给鸭子类型一个"可检查的契约"
from typing import Protocol, runtime_checkable

@runtime_checkable                       # 开启后可用 isinstance 做结构化检查
class Runnable(Protocol):
    def run(self) -> None: ...

class Task:                              # 注意：没有继承 Runnable！
    def run(self):
        print("Task.run")

print("Task 满足 Runnable 协议:", isinstance(Task(), Runnable))   # True —— 只看结构

def execute(obj: Runnable) -> None:      # 静态检查器（mypy）也能据此校验
    obj.run()

execute(Task())    # Task.run

Task 满足 Runnable 协议: True
Task.run


In [50]:
# 【9.4】__getattr__ 实现代理：把不认识的属性转发给目标对象
class Proxy:
    def __init__(self, target):
        self._target = target

    def __getattr__(self, name):
        return getattr(self._target, name)   # 转发

p = Proxy([1, 2, 3])
print("append 的类型:", type(p.append))   # <class 'builtin_function_or_method'>
# （不直接 print(p.append)：其 repr 含内存地址，输出不可复现）
p.append(4)                               # 实际执行的是 list.append
print("透传结果:", p._target)              # [1, 2, 3, 4]

append 的类型: <class 'builtin_function_or_method'>
透传结果: [1, 2, 3, 4]


# 第 10 章 动态函数

> **本章解决什么问题**：函数也是对象——可以动态创建（闭包）、携带属性、被字符串索引和调用（路由/回调的原理）。

In [51]:
# 【10.1】闭包：工厂函数动态"生产"函数
def make_adder(n):
    def adder(x):
        return x + n       # 捕获外层变量 n —— 每个 adder 都带着自己的 n
    return adder

add5 = make_adder(5)
add10 = make_adder(10)
print(add5(10))    # 15
print(add10(10))   # 20

15
20


In [52]:
# 【10.2】函数可以携带任意属性（函数本身就是对象）
def handler():
    pass

handler.version = "1.0"          # 给函数挂元数据
handler.role = "callback"
print(handler.version, handler.role)   # 1.0 callback
print(vars(handler))                    # 函数也有 __dict__

1.0 callback
{'version': '1.0', 'role': 'callback'}


In [53]:
# 【10.3】inspect.signature：运行时解析函数签名（依赖注入的基础）
import inspect

def f(a, b=1, *args, **kwargs):
    pass

sig = inspect.signature(f)
for name, param in sig.parameters.items():
    print(f"参数 {name}: kind={param.kind.name}, default={param.default!r}")
# 预期输出：
# 参数 a: POSITIONAL_OR_KEYWORD, default=<class 'inspect._empty'>（无默认值）
# 参数 b: POSITIONAL_OR_KEYWORD, default=1
# 参数 args: VAR_POSITIONAL, default=<class 'inspect._empty'>
# 参数 kwargs: VAR_KEYWORD, default=<class 'inspect._empty'>

参数 a: kind=POSITIONAL_OR_KEYWORD, default=<class 'inspect._empty'>
参数 b: kind=POSITIONAL_OR_KEYWORD, default=1
参数 args: kind=VAR_POSITIONAL, default=<class 'inspect._empty'>
参数 kwargs: kind=VAR_KEYWORD, default=<class 'inspect._empty'>


In [54]:
# 【10.4】按字符串动态获取并调用函数（路由表 / 回调注册的底层）
import math

func = getattr(math, "sqrt")          # 字符串 → 模块属性（函数）
print(func(16))                        # 4.0

def my_handler():
    return "from globals"

func2 = globals()["my_handler"]        # 字符串 → 当前命名空间
print(func2())                         # from globals

4.0
from globals


# 第 11 章 动态模块

> **本章解决什么问题**：模块也是对象——可以运行时添加属性、甚至凭空创建模块对象（测试桩 / 兼容层常用）。

In [55]:
# 【11.1】给已导入的模块动态添加属性
import math

math.custom_tag = "added-by-notebook"   # 模块对象上直接挂属性
print(math.custom_tag)                  # added-by-notebook

del math.custom_tag                     # 用完清理，保持干净
print("已清理:", not hasattr(math, "custom_tag"))   # True

added-by-notebook
已清理: True


In [56]:
# 【11.2】运行时创建一个完整的模块对象（mock 依赖 / 兼容层）
import types, sys

mod = types.ModuleType("mymodule", "运行时创建的演示模块")
mod.x = 10
mod.greet = lambda: "hello"

sys.modules["mymodule"] = mod   # 注册进模块缓存 → import 语句可以"导入"它
import mymodule
print(mymodule.x, mymodule.greet())   # 10 hello

del sys.modules["mymodule"]     # 清理，避免影响后续导入
print("已从 sys.modules 移除:", "mymodule" not in sys.modules)   # True

10 hello
已从 sys.modules 移除: True


> ⚠️ **修正说明**：源文档 11.1 的 `import module` 是无法运行的伪代码，此处改为对真实模块 `math` 操作；
> 11.3 的 `__all__` 示例依赖一个磁盘上的模块文件，下一节用临时目录中的真实模块验证。

In [57]:
# 【11.3】__all__：控制 from module import * 的导出范围（真实模块验证）
import os, sys

_mod_path = os.path.join(_plug_root, "sample_api.py")
with open(_mod_path, "w") as f:
    f.write(
        "__all__ = ['api_a', 'api_b']\n\n"
        "def api_a(): return 'a'\n\n"
        "def api_b(): return 'b'\n\n"
        "def api_c(): return 'c'\n"
    )

sys.modules.pop("sample_api", None)   # 防止重跑时命中缓存
from sample_api import *              # 只导入 __all__ 列出的名字

print("api_a():", api_a())                    # a
print("api_b 已导入:", "api_b" in dir())      # True
print("api_c 已导入:", "api_c" in dir())      # False —— 被 __all__ 排除

api_a(): a
api_b 已导入: True
api_c 已导入: False


## 11.4 使用场景

动态配置 / 插件 / 测试（伪造依赖模块）/ 兼容层（旧 API 映射到新实现）。

# 第 12 章 反例单元：错误写法 vs 正确写法

> **本章解决什么问题**：动态特性最容易写错的地方，先看**真实报错**，再看正确姿势与输出差异。
> （为保证「报错单元数 = 0」，错误示例均用 try/except 捕获并打印异常——这本身就是标准的错误演示方式。）

## 反例 1：`__getattr__` 内部访问自身属性 → 无限递归

In [58]:
# ---------- 错误写法：__getattr__ 里访问 self.name，而 name 不存在 → 再次触发 __getattr__
class Bad:
    def __getattr__(self, name):
        return self.name        # self.name 找不到 → 又进 __getattr__ → 无限递归

try:
    Bad().anything
except RecursionError as e:
    print("错误写法报错:", type(e).__name__)   # RecursionError

# ---------- 正确写法 1：明确抛 AttributeError（让"真找不到"正常失败）
class Good1:
    def __getattr__(self, name):
        raise AttributeError(f"{type(self).__name__} 没有属性 {name!r}")

try:
    Good1().anything
except AttributeError as e:
    print("正确写法 1 报错:", e)   # 清晰、可调试

# ---------- 正确写法 2：确实需要兜底时，绕过协议直接查 __dict__
class Good2:
    def __getattr__(self, name):
        return self.__dict__.get(name, "默认值")   # 直接查字典，不再触发协议

print(Good2().anything)    # 默认值

错误写法报错: RecursionError
正确写法 1 报错: Good1 没有属性 'anything'
默认值


## 反例 2：`__getattribute__` 内部访问属性 → 死循环

In [59]:
# ---------- 错误写法：__getattribute__ 拦截【所有】访问，内部再访问 self.x → 再次拦截 → 死循环
class BadAttr:
    def __getattribute__(self, name):
        return self.x        # self.x 又触发 __getattribute__……

try:
    BadAttr().x
except RecursionError:
    print("错误写法报错: RecursionError")

# ---------- 正确写法：无条件转发给 object 的默认实现
class GoodAttr:
    def __getattribute__(self, name):
        print(f"  [审计] {name}")                       # 自定义逻辑
        return super().__getattribute__(name)           # 关键：转发

g = GoodAttr()
g.x = 1
print(g.x)    # 正常输出：审计日志 + 1

错误写法报错: RecursionError
  [审计] x
1


## 反例 3：猴子补丁内置类型

> ⚠️ **修正说明**：源文档 13.4 称 `list.append = lambda self, x: None` 会"破坏内置类型"。
> 实际上 **CPython 直接禁止给内置类型赋属性**，该语句会立即抛 `TypeError`——这是解释器的保护机制，验证如下。

In [60]:
# ---------- 错误写法：给内置类型打补丁 → 直接被解释器拦截
try:
    list.append = lambda self, x: None
except TypeError as e:
    print("错误写法报错:", e)
    # 'cannot set 'append' attribute of immutable type 'list''

# ---------- 正确姿势：想扩展内置类型 → 继承它
class SafeList(list):
    def append(self, x):
        print(f"  [记录] append({x!r})")
        super().append(x)

sl = SafeList([1])
sl.append(2)          #   [记录] append(2)
print(sl)             # [1, 2]

错误写法报错: cannot set 'append' attribute of immutable type 'list'
  [记录] append(2)
[1, 2]


## 反例 4：`__getattr__` 兜底吞掉拼写错误

In [61]:
# ---------- 错误写法：无条件返回默认值 → 拼写错误永远不报错
class TypoTrap:
    def __getattr__(self, name):
        return 1

t = TypoTrap()
print("拼写错误也返回:", t.anything_tpyo)   # 1 —— 本想访问 anything_typo，静默拿错值

# ---------- 正确写法：白名单 + 明确报错
class Strict:
    _allowed = {"debug", "verbose"}
    def __getattr__(self, name):
        if name in self._allowed:
            return False
        raise AttributeError(f"未知配置项 {name!r}，可选: {sorted(self._allowed)}")

s = Strict()
print("合法项:", s.debug)    # False
try:
    s.dbug                   # 拼写错误
except AttributeError as e:
    print("拼写错误立即暴露:", e)

拼写错误也返回: 1
合法项: False
拼写错误立即暴露: 未知配置项 'dbug'，可选: ['debug', 'verbose']


## 反例 5：动态导入未校验模块名（路径注入）

In [62]:
# ---------- 错误写法：用户输入直接拼进 import → 可导入任意模块（信息泄露/攻击面）
import importlib

user_input = "os"          # 假装这是外部输入
# module = importlib.import_module(user_input)  # 危险：攻击者可传入 os / sys 等
print("错误写法：未校验即导入", user_input, "→ 危险，已注释")

# ---------- 正确写法：白名单前缀校验
ALLOWED_PLUGIN_PREFIX = "plugins."

def safe_import_plugin(name):
    if not name.startswith(ALLOWED_PLUGIN_PREFIX):
        raise ImportError(f"只允许导入 {ALLOWED_PLUGIN_PREFIX}* 模块，收到: {name!r}")
    return importlib.import_module(name)

mod = safe_import_plugin("plugins.pdf")       # 合法
print("合法导入成功:", mod.register()["name"])   # pdf
try:
    safe_import_plugin("os")                  # 非法
except ImportError as e:
    print("非法导入被拦截:", e)

错误写法：未校验即导入 os → 危险，已注释
合法导入成功: pdf
非法导入被拦截: 只允许导入 plugins.* 模块，收到: 'os'


## 反例 6：`hasattr` + `getattr` 双重查找（性能 & 竞态）

In [63]:
# ---------- 错误写法：先 hasattr 再 getattr —— 属性被查找【两次】
class Probe:
    def __getattr__(self, name):
        print(f"  [触发 __getattr__] {name}")     # 计数：__getattr__ 被调用几次？
        return 42

p = Probe()
if hasattr(p, "key"):        # 第 1 次查找（触发 __getattr__）
    v1 = getattr(p, "key")   # 第 2 次查找（又触发 __getattr__）
print("错误写法取值:", v1)

# ---------- 正确写法：getattr 带默认值 —— 只查找【一次】，语义相同
p2 = Probe()
v2 = getattr(p2, "key", None)   # 仅 1 次触发
print("正确写法取值:", v2)
# 输出对比：错误写法打印两条 [触发] 日志，正确写法只有一条

  [触发 __getattr__] key
  [触发 __getattr__] key
错误写法取值: 42
  [触发 __getattr__] key
正确写法取值: 42


# 第 13 章 实用示例集：5 个工程高频模式

> **本章解决什么问题**：把前面的知识点组装成生产代码里天天见的模式（以 LLMOps API 服务为背景）。

| # | 模式 | 核心知识点 | 典型场景 |
|---|---|---|---|
| 13.1 | 动态配置对象 | `__dict__` / `setattr` 批量 / `__setattr__` 校验 | 把 YAML/环境变量变成对象 |
| 13.2 | ORM 风格字段 | 描述器 + `__set_name__` | 模型定义、参数模式 |
| 13.3 | 链式 API 客户端 | `__getattr__` | SDK 封装远端资源 |
| 13.4 | Mock 对象 | `__getattr__` + 记录调用 | 测试桩 |
| 13.5 | 动态路由 | 装饰器 + 字典路由表 | Web 框架 / 任务分发 |

## 13.1 动态配置对象（含参数校验）

**场景**：LLM 推理参数来自 YAML，要挂成对象属性并校验取值范围。
**注意点**：`__setattr__` 校验要在 `__init__` 赋值时也生效（统一走 `setattr`）；序列化时过滤私有属性。

In [64]:
# 【13.1】动态配置对象：批量挂载 + 校验 + 序列化
class ModelConfig:
    # LLM 推理参数配置：动态属性 + 赋值校验 + 序列化

    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)          # 统一走 __setattr__ → 校验对构造同样生效

    def __setattr__(self, name, value):
        if name == "temperature":
            if not (isinstance(value, (int, float)) and 0 <= value <= 2):
                raise ValueError(f"temperature 必须在 [0, 2]，收到 {value!r}")
        super().__setattr__(name, value)

    def to_dict(self):                   # 序列化：过滤私有属性
        return {k: v for k, v in self.__dict__.items() if not k.startswith("_")}

cfg = ModelConfig(model="demo-1", temperature=0.7, _internal="隐藏")
print("读取:", cfg.model, cfg.temperature)       # demo-1 0.7
print("序列化:", cfg.to_dict())                   # {'model': 'demo-1', 'temperature': 0.7}

try:
    cfg.temperature = 5                  # 越界
except ValueError as e:
    print("校验生效:", e)

读取: demo-1 0.7
序列化: {'model': 'demo-1', 'temperature': 0.7}
校验生效: temperature 必须在 [0, 2]，收到 5


## 13.2 ORM 风格字段（描述器）

**场景**：声明式定义数据模型，字段名自动绑定、可扩展类型校验。
**注意点**：描述器是**数据描述器**（定义 `__set__`），优先级高于实例 `__dict__`，所以 `__set__` 里必须把值写进 `obj.__dict__[self.name]`；`__set_name__` 在类创建时自动拿到字段名，无需重复书写。

In [65]:
# 【13.2】Field 描述器 + Model 基类：声明式模型
class Field:
    # 数据描述器：拦截字段读写，自动感知字段名

    def __set_name__(self, owner, name):   # 类创建时自动回调（知识点 4.4 的近亲）
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:                    # 类上访问（User.name）→ 返回描述器自身
            return self
        return obj.__dict__.get(self.name) # 实例上访问 → 取存储的值

    def __set__(self, obj, value):
        obj.__dict__[self.name] = value    # 存进实例 __dict__（键与字段同名）

class Model:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

class User(Model):
    id = Field()          # 声明字段
    name = Field()

u = User(id=1, name="Alice")
print(u.id, u.name)        # 1 Alice
print("存储位置:", vars(u))  # {'id': 1, 'name': 'Alice'}

1 Alice
存储位置: {'id': 1, 'name': 'Alice'}


## 13.3 链式 API 客户端（`__getattr__`）

**场景**：远端资源种类多且会增长（`client.chat_completions`、`client.embeddings`……），不想每加一个资源就写一个方法。
**注意点**：`__getattr__` 里返回的闭包每次都会新建；只对**未知**属性生效，已定义的方法不受影响。

In [66]:
# 【13.3】LLMClient：资源名即方法名
class LLMClient:
    def __init__(self, base_url):
        self.base_url = base_url

    def __getattr__(self, name):          # 只在常规查找失败时触发
        def method(**kwargs):             # 惰性生成"资源方法"
            return f"[{self.base_url}] 调用资源 {name}，参数 {kwargs}"
        return method

client = LLMClient("https://api.llm.internal")
print(client.chat_completions(model="demo-1", temperature=0.7))
print(client.embeddings(input="你好"))
# 新增远端资源【零代码改动】—— 新名字自动变成方法

[https://api.llm.internal] 调用资源 chat_completions，参数 {'model': 'demo-1', 'temperature': 0.7}
[https://api.llm.internal] 调用资源 embeddings，参数 {'input': '你好'}


## 13.4 Mock 对象（记录一切调用）

**场景**：测试中替代真实依赖（如 LLM API），断言调用参数。
**注意点**：`calls` 列表在 `__init__` 里设置，位于实例 `__dict__`，不会误触发 `__getattr__`（验证 0.4 节查找顺序）。

In [67]:
# 【13.4】Mock：任何方法调用都被记录
class Mock:
    def __init__(self):
        self.calls = []                   # 实例 __dict__ 中 → 访问不触发 __getattr__

    def __getattr__(self, name):          # 未知"方法名" → 生成记录器
        def method(*args, **kwargs):
            self.calls.append((name, args, kwargs))
            return None
        return method

m = Mock()
m.foo(1, 2)
m.bar(x=3)
print(m.calls)
# [('foo', (1, 2), {}), ('bar', (), {'x': 3})] —— 可直接用于断言

[('foo', (1, 2), {}), ('bar', (), {'x': 3})]


## 13.5 动态路由（装饰器 + 路由表）

**场景**：API 端点 / 任务处理器注册。装饰器把函数登记进字典，`dispatch` 按字符串分发。
**注意点**：路由未命中要抛 `KeyError` 而非静默返回 `None`；装饰器必须 `return func` 保持函数可用。

In [68]:
# 【13.5】Router：装饰器注册 + dispatch 分发
class Router:
    def __init__(self):
        self.routes = {}

    def route(self, path):
        def decorator(func):
            self.routes[path] = func      # 注册：路径 → 处理函数
            return func                   # 关键：原样返回，不破坏被装饰函数
        return decorator

    def dispatch(self, path, *args, **kwargs):
        if path not in self.routes:
            raise KeyError(f"未注册的路由: {path}")
        return self.routes[path](*args, **kwargs)

router = Router()

@router.route("/")
def home():
    return "home"

@router.route("/users/<id>")
def users(id):
    return f"user-{id}"

print(router.dispatch("/"))                    # home
print(router.dispatch("/users/<id>", id=42))   # user-42
try:
    router.dispatch("/missing")
except KeyError as e:
    print("未命中:", e)

home
user-42
未命中: '未注册的路由: /missing'


# 第 14 章 内置 / 生态能力对照

> **本章解决什么问题**：动态特性的"官方工具箱"和"第三方替代品"一张表看全，选型不纠结。

## 标准库

| 能力 | 工具 | 说明 |
|---|---|---|
| 反射式属性操作 | `getattr / setattr / hasattr / delattr` | 内置函数，字符串操作属性 |
| 命名空间 / 类型 | `vars / dir / type / isinstance` | 内置函数 |
| 动态方法 / 模块 | `types.MethodType / types.ModuleType` | `types` 模块 |
| 内省 | `inspect` | 签名、源码、调用栈、参数类别 |
| 动态导入 | `importlib` + `pkgutil` | 字符串导入、插件发现、reload |
| 安全解析 | `ast.literal_eval` | 只认字面量，拒绝代码 |
| 结构化契约 | `typing.Protocol`（3.8+） | 鸭子类型的静态可检查版 |
| 类型分派 | `functools.singledispatch` | 按参数类型自动选择实现 |
| 测试替身 | `unittest.mock` | 自动管理的猴子补丁 |

## 生态（第三方）

| 库 | 定位 | 与动态特性的关系 |
|---|---|---|
| `dataclasses`（标准库） | 声明式数据类 | 用类装饰器 + 字段描述替代手写 `__init__` |
| `pydantic` | 数据校验 | 生产级替代手写 `__setattr__` 校验 |
| `attrs` | 类增强 | 描述器自动生成 |
| `pluggy` | 插件框架 | pytest 同款：钩子 + 动态导入的工程化封装 |

In [69]:
# 【14】functools.singledispatch：按类型自动分派（最佳实践第 11 条）
from functools import singledispatch

@singledispatch
def serialize(obj):
    return str(obj)                       # 兜底实现

@serialize.register
def _(obj: dict):                         # 按【类型注解】注册
    return {k: serialize(v) for k, v in obj.items()}

@serialize.register
def _(obj: list):
    return [serialize(v) for v in obj]

@serialize.register
def _(obj: bool):                         # bool 必须注册在 int 之前才优先命中
    return "true" if obj else "false"

print(serialize({"enabled": True, "tags": [1, "a"], "note": None}))
# {'enabled': 'true', 'tags': ['1', 'a'], 'note': 'None'}

{'enabled': 'true', 'tags': ['1', 'a'], 'note': 'None'}


# 第 15 章 综合实战：可扩展的 LLM 网关

> **本章解决什么问题**：把第 1~13 章的知识点组装成一个贴近 LLMOps 业务的迷你框架——
> **Provider 插件化注册（Protocol + `__init_subclass__` + 动态导入）+ 参数校验（`__setattr__`）+
> 资源式调用（`__getattr__`）+ 路由分发（装饰器路由表）**。全程本地 mock，无网络请求，输出确定。

## 架构图（文字版）

```
                ┌────────────────────────────────────────┐
   HTTP-ish ──► │  Router（13.5 路由表，装饰器注册端点）   │
   路由请求     └───────────────┬────────────────────────┘
                                ▼
                ┌────────────────────────────────────────┐
                │  Gateway（调度器）                       │
                │  · providers: 内置注册表 + 外部插件      │
                │  · dispatch(provider, prompt, params)  │
                └───────┬──────────────────────┬─────────┘
                        ▼                      ▼
        ┌──────────────────────┐   ┌─────────────────────────┐
        │ 内置 Provider         │   │ llmops_ext 插件包        │
        │ Echo / Upper          │   │ reverse_provider.py     │
        │ （__init_subclass__    │   │ （pkgutil 发现 +         │
        │   自动注册）           │   │   importlib 动态导入）   │
        └──────────────────────┘   └─────────────────────────┘
                        ▲
        参数统一走 ModelConfig（13.1 __setattr__ 校验）
```

## Part 1：契约 + 自动注册（知识点：Protocol、`__init_subclass__`、类型校验）

In [70]:
# 【15.1】Provider 契约与自动注册
from typing import Protocol, runtime_checkable

@runtime_checkable
class Provider(Protocol):                 # 契约：插件必须"长得像"Provider
    name: str
    def invoke(self, prompt: str, **params) -> str: ...

class BaseProvider:
    registry: dict = {}                   # 类级注册表

    def __init_subclass__(cls, **kwargs): # 每定义一个子类 → 自动登记
        super().__init_subclass__(**kwargs)
        if not callable(getattr(cls, "invoke", None)):
            raise TypeError(f"{cls.__name__} 必须实现 invoke()")
        BaseProvider.registry[cls.name] = cls

class EchoProvider(BaseProvider):
    name = "echo"
    def invoke(self, prompt, **params):
        return f"[echo] {prompt}"

class UpperProvider(BaseProvider):
    name = "upper"
    def invoke(self, prompt, **params):
        return f"[upper] {prompt.upper()}"

print("内置 Provider:", sorted(BaseProvider.registry))   # ['echo', 'upper']

# 验证：不实现 invoke 的子类在【定义时】就被拒绝
try:
    class BrokenProvider(BaseProvider):
        name = "broken"
        # 没有 invoke
except TypeError as e:
    print("注册校验:", e)

内置 Provider: ['echo', 'upper']
注册校验: BrokenProvider 必须实现 invoke()


## Part 2：外部插件动态加载（知识点：临时模块包、`pkgutil` 发现、`importlib` 导入、鸭子类型校验）

In [71]:
# 【15.2】在临时目录创建外部插件包（模拟 pip 安装的第三方扩展）
import os, sys, tempfile

_gw_root = tempfile.mkdtemp(prefix="nb_gw_")
_ext = os.path.join(_gw_root, "llmops_ext")
os.makedirs(_ext)
open(os.path.join(_ext, "__init__.py"), "w").close()

plugin_code = (
    "def register():\n"
    "    class ReverseProvider:\n"          # 注意：不继承 BaseProvider！
    "        name = 'reverse'\n"
    "        def invoke(self, prompt, **params):\n"
    "            return f'[reverse] {prompt[::-1]}'\n"
    "    return ReverseProvider()\n"
)
with open(os.path.join(_ext, "reverse_provider.py"), "w") as f:
    f.write(plugin_code)

sys.path.insert(0, _gw_root)
print("插件包已就绪:", sorted(m for m in os.listdir(_ext) if m.endswith(".py")))

插件包已就绪: ['__init__.py', 'reverse_provider.py']


In [72]:
# 【15.3】Gateway：整合内置注册表 + 动态发现外部插件
import importlib, pkgutil

class Gateway:
    def __init__(self, ext_packages=()):
        # 内置注册表存的是【类】→ 实例化后统一管理；外部插件 register() 已返回实例
        self.providers = {name: cls() for name, cls in BaseProvider.registry.items()}
        for pkg_name in ext_packages:
            self._load_extensions(pkg_name)

    def _load_extensions(self, pkg_name):
        # 知识点 7.5：pkgutil 发现 + importlib 导入（sorted 保证可复现）
        pkg = importlib.import_module(pkg_name)
        for _, mod_name, _ in sorted(pkgutil.iter_modules(pkg.__path__)):
            module = importlib.import_module(f"{pkg_name}.{mod_name}")
            if hasattr(module, "register"):            # 插件约定：暴露 register()
                provider = module.register()
                # 知识点 9.3：Protocol 结构化校验（鸭子类型的"契约化"）
                if not isinstance(provider, Provider):
                    raise TypeError(f"{mod_name} 返回的对象不满足 Provider 协议")
                self.providers[provider.name] = provider

    def dispatch(self, provider_name, prompt, **params):
        if provider_name not in self.providers:
            raise KeyError(f"未注册的 provider: {provider_name}")
        return self.providers[provider_name].invoke(prompt, **params)

    def __getattr__(self, name):
        # 知识点 3.5/13.3：未知属性给出【带上下文】的报错，而非裸 AttributeError
        raise AttributeError(
            f"Gateway 没有属性 {name!r}；可用 provider: {sorted(self.providers)}")

gw = Gateway(ext_packages=("llmops_ext",))
print("全部 Provider:", sorted(gw.providers))   # ['echo', 'reverse', 'upper']

全部 Provider: ['echo', 'reverse', 'upper']


## Part 3：路由 + 参数校验，端到端跑通（知识点：13.1 / 13.5 组合）

In [73]:
# 【15.4】端到端：路由分发 → 网关调度 → 参数校验
api = Router()                      # 13.5 的 Router

@api.route("/v1/echo")
def echo_endpoint(prompt, **params):
    return gw.dispatch("echo", prompt, **params)

@api.route("/v1/reverse")
def reverse_endpoint(prompt, **params):
    return gw.dispatch("reverse", prompt, **params)

cfg = ModelConfig(model="demo-1", temperature=0.5)   # 13.1 的校验配置

print(api.dispatch("/v1/echo", prompt="你好，动态特性"))
print(api.dispatch("/v1/reverse", prompt="stressed", temperature=cfg.temperature))
# [reverse] desserts —— "stressed" 反转

# 未知 provider / 未知属性 → 清晰报错
try:
    gw.dispatch("nonexistent", "hi")
except KeyError as e:
    print("调度报错:", e)
try:
    gw.some_typo
except AttributeError as e:
    print("属性报错:", e)

[echo] 你好，动态特性
[reverse] desserts
调度报错: '未注册的 provider: nonexistent'
属性报错: Gateway 没有属性 'some_typo'；可用 provider: ['echo', 'reverse', 'upper']


## 设计取舍（为什么这么做）

| 决策 | 备选方案 | 取舍理由 |
|---|---|---|
| `__init_subclass__` 做注册 | 元类 | 需求只是"登记子类"，元类过度设计（最佳实践 4） |
| 插件用 `register()` 工厂 + Protocol 校验 | 强制继承 BaseProvider | 插件与框架**解耦**：第三方无需 import 框架代码也能写插件；Protocol 只验结构 |
| `pkgutil` 发现 + `sorted` | 显式配置模块清单 | 新增插件零配置；sorted 保证加载顺序可复现 |
| `__setattr__` 手写校验 | pydantic | 教学演示动态协议；**生产建议 pydantic**（生态对照 14 章） |
| `__getattr__` 抛带上下文的 AttributeError | 裸抛 | 避免反例 4 的"静默吞错"，把可用 provider 列表带进报错 |
| dispatch 用 `KeyError` | 返回 None | 显式失败优于静默空值（EAFP 风格） |

# 第 16 章 常见陷阱汇总（陷阱 / 症状 / 解法）

> **本章解决什么问题**：一张表带走全部高频坑，每条都能在本章或第 12 章找到可运行的复现代码。

| # | 陷阱 | 症状 | 解法（复现位置） |
|---|---|---|---|
| 1 | 动态属性难以发现 | IDE 无提示、拼写错误不报错 | 白名单 + 显式 AttributeError（12 章反例 4） |
| 2 | `__getattr__` 无限递归 | `RecursionError` | 访问 `self.__dict__` 或抛 AttributeError（12 章反例 1） |
| 3 | `__getattribute__` 死循环 | `RecursionError` | 内部一律 `super().__getattribute__`（12 章反例 2） |
| 4 | 猴子补丁内置类型 | `TypeError: cannot set ...`（被拦截） | 继承扩展，别补丁内置类型（12 章反例 3） |
| 5 | `eval` / `exec` 不可信输入 | 任意代码执行漏洞 | AST 白名单 / `literal_eval`（8 章） |
| 6 | 动态导入无校验 | 路径注入、任意模块加载 | 白名单前缀（12 章反例 5） |
| 7 | 类型注解不强制 | 注解 int 实传 str，运行照常 | mypy/IDE 静态检查（1.3 节验证） |
| 8 | 动态访问性能开销 | `getattr` 慢于点号访问 | 热路径用点号；hasattr+getattr 合并为一次 getattr（12 章反例 6） |
| 9 | 动态类无法 pickle | `PicklingError` / `AttributeError` | 类放模块顶层、名字与 qualname 一致（下方复现） |
| 10 | 动态生成代码难调试 | 断点打不进去、栈里是 `<string>` | `compile` 指定文件名、加日志、限定使用范围 |

In [74]:
# 【16.1】复现陷阱 1：拼写错误被 __getattr__ 静默吞掉
class AnythingGoes:
    def __getattr__(self, name):
        return 1

a = AnythingGoes()
print("拼写错误也能取到值:", a.anything_tpyo)   # 1 —— 本想写 anything_typo
# 解法见 12 章反例 4：白名单 + 显式报错

拼写错误也能取到值: 1


In [75]:
# 【16.2】复现陷阱 9：函数内动态创建的类无法 pickle
import pickle

def make_class():
    return type("Local", (), {"greet": lambda self: "hi"})

LocalCls = make_class()          # qualname 是 "Local"，但 __main__ 里查不到这个名字
try:
    pickle.dumps(LocalCls)
except Exception as e:
    print(f"{type(e).__name__}: {e}")
    # pickle 按 "模块.类名" 反查 —— 查不到 Local → 失败
# 解法：动态类放到模块顶层，让名字可被反查到

PicklingError: Can't pickle <class '__main__.Local'>: attribute lookup Local on __main__ failed


In [76]:
# 【16.3】复现陷阱 8：动态属性访问的性能开销
# 注：耗时数值因机器而异，但【相对结论】稳定 —— 直接访问恒快于 getattr
import timeit

setup = "class C:\n    x = 1\nobj = C()"
t_direct = min(timeit.repeat("obj.x", setup=setup, number=1_000_000, repeat=5))
t_getattr = min(timeit.repeat("getattr(obj, 'x')", setup=setup, number=1_000_000, repeat=5))

print(f"点号访问   : {t_direct:.3f}s / 百万次")
print(f"getattr 访问: {t_getattr:.3f}s / 百万次")
print("结论：点号访问更快 =", t_direct < t_getattr)   # True

点号访问   : 0.005s / 百万次


getattr 访问: 0.013s / 百万次
结论：点号访问更快 = True


In [77]:
# 【16.4】复现陷阱 10：动态执行代码的调用栈难以定位
import traceback
try:
    exec("1 / 0", {"__builtins__": __builtins__})   # exec 里的除零
except ZeroDivisionError:
    tb = traceback.format_exc().splitlines()
    print("栈顶文件名:", tb[-2].strip())   # 文件名显示 <string> —— 调试器很难定位
# 解法：compile 时传入有意义的文件名，或避免对业务代码使用 exec

栈顶文件名: File "<string>", line 1, in <module>


In [78]:
# 【16.5】清理本 Notebook 创建的临时模块与 sys.path（保持环境干净）
import sys

for _name in list(sys.modules):
    if (_name == "plugins" or _name.startswith("plugins.")
            or _name == "llmops_ext" or _name.startswith("llmops_ext.")
            or _name in ("sample_api", "mymodule")):
        del sys.modules[_name]

for _p in (_plug_root, _gw_root):
    if _p in sys.path:
        sys.path.remove(_p)

print("临时模块与 sys.path 已清理（临时目录由操作系统自动回收）")

临时模块与 sys.path 已清理（临时目录由操作系统自动回收）


# 第 17 章 总结

## 17.1 动态特性全景图

| 类别 | 机制 | 对应章节 |
|---|---|---|
| 动态类型 | 变量无类型，运行时绑定 | 第 1 章 |
| 动态属性 | `setattr`、`__getattr__`、`__dict__` | 第 2 章 |
| 属性访问协议 | `__getattr__` / `__getattribute__` / `__setattr__` / `__delattr__` | 第 3 章 |
| 动态类 | `type()`、元类、`__init_subclass__`、类装饰器 | 第 4 章 |
| 反射 | `dir`、`vars`、`inspect`、`getattr` | 第 5 章 |
| 猴子补丁 | 运行时修改类 / 模块 / 实例 | 第 6 章 |
| 动态导入 | `importlib`、`pkgutil`、reload | 第 7 章 |
| 动态执行 | `eval`、`exec`、`compile` | 第 8 章 |
| 动态协议 | 鸭子类型、魔术方法、`Protocol` | 第 9 章 |
| 动态函数 | 闭包、函数属性、签名、动态调用 | 第 10 章 |
| 动态模块 | `types.ModuleType`、`sys.modules`、`__all__` | 第 11 章 |

## 17.2 核心权衡

- **灵活性 vs 可维护性**：动态越少，新人越容易看懂
- **简洁 vs 可读性**：一行 `type()` 顶几十行显式定义，但 IDE 追踪不到
- **动态 vs 类型安全**：mypy 只能检查注解过的部分
- **强大 vs 安全风险**：`eval` 一开，攻击面全开

## 17.3 通用代码模板（可直接复制）

```python
# ---------- 模板 1：代理对象 ----------
class Proxy:
    def __init__(self, target):
        self._target = target
    def __getattr__(self, name):               # 只拦"找不到"的
        return getattr(self._target, name)

# ---------- 模板 2：自动注册基类 ----------
class Base:
    registry: dict = {}
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Base.registry[cls.__name__] = cls

# ---------- 模板 3：带校验的配置对象 ----------
class Config:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)                # 统一走校验
    def __setattr__(self, name, value):
        # ... 按需校验 ...
        super().__setattr__(name, value)

# ---------- 模板 4：插件加载器 ----------
import importlib, pkgutil

def load_plugins(package, marker="register"):
    pkg = importlib.import_module(package)
    found = {}
    for _, name, _ in sorted(pkgutil.iter_modules(pkg.__path__)):
        module = importlib.import_module(f"{package}.{name}")
        if hasattr(module, marker):
            found[name] = getattr(module, marker)()
    return found

# ---------- 模板 5：AST 白名单求值 ----------
import ast
ALLOWED = (ast.Expression, ast.BinOp, ast.Constant, ast.UnaryOp,
           ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub)

def safe_eval(expr):
    tree = ast.parse(expr, mode="eval")
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED):
            raise ValueError(f"不支持节点: {type(node).__name__}")
    return eval(compile(tree, "<safe>", "eval"))
```

## 17.4 记忆口诀

1. **变量是名字，对象才有型**
2. **属性找不到，才进 `__getattr__`**
3. **改类用钩子（`__init_subclass__` / 装饰器），少碰元类**
4. **动态导入白名单，eval 永不碰输入**
5. **补丁只在测试里，生产代码要显式**
6. **鸭子看行为，Protocol 写契约**

## 17.5 自测练习（先自己写，再看下节参考实现）

1. 实现 `Cached` 类：访问任意未定义属性返回 `"default:<name>"`，且第二次访问不再触发 `__getattr__`
2. 用 `type()` 动态创建 `Pair` 类，含 `first`/`second` 属性和 `swap()` 方法
3. 实现 `load_class("json.JSONDecoder")`：从字符串导入并返回类，要求校验模块白名单
4. 用 `__init_subclass__` 实现 `Handler` 基类：子类以 `key="..."` 参数注册，支持按 key 查找
5. 扩展 `safe_eval`：支持一元负号与幂运算 `**`，并验证 `-2 ** 2` 与 `(-2) ** 2` 的结果差异

## 17.6 参考实现

In [79]:
# 【17.6】自测练习参考实现

# ---------- 练习 1：Cached（考点：__getattr__ + __dict__ 缓存）
class Cached:
    def __getattr__(self, name):
        value = f"default:{name}"
        setattr(self, name, value)            # 写入 __dict__ → 下次不触发
        return value

c = Cached()
print(c.foo, "| 第二次:", c.foo, "| 已缓存:", "foo" in vars(c))
# default:foo | 第二次: default:foo | 已缓存: True

# ---------- 练习 2：type() 创建 Pair（考点：动态类）
Pair = type("Pair", (), {"swap": lambda self: (self.second, self.first)})
p = Pair()
p.first, p.second = 1, 2
print("swap:", p.swap())                     # (2, 1)

# ---------- 练习 3：白名单 load_class（考点：动态导入 + 安全校验）
import importlib
ALLOWED_PREFIXES = ("json", "os.path")

def load_class(path):
    module_name, _, class_name = path.rpartition(".")
    if not any(module_name == pre or module_name.startswith(pre + ".")
               for pre in ALLOWED_PREFIXES):
        raise ImportError(f"模块不在白名单: {module_name}")
    module = importlib.import_module(module_name)
    return getattr(module, class_name)

print("加载类:", load_class("json.JSONDecoder"))
try:
    load_class("sys.path")                    # 不在白名单
except ImportError as e:
    print("拦截:", e)

# ---------- 练习 4：带 key 的注册基类（考点：__init_subclass__ 关键字参数）
class Handler:
    _registry = {}
    def __init_subclass__(cls, key=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if key is None:
            raise TypeError("子类必须提供 key='...'")
        Handler._registry[key] = cls
    @classmethod
    def get(cls, key):
        return cls._registry[key]

class SyncHandler(Handler, key="sync"): pass
print("按 key 取类:", Handler.get("sync") is SyncHandler)   # True

# ---------- 练习 5：支持负号与幂的 safe_eval（考点：AST 白名单）
import ast
ALLOWED = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
           ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub)

def safe_eval2(expr):
    tree = ast.parse(expr, mode="eval")
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED):
            raise ValueError(f"不支持节点: {type(node).__name__}")
    return eval(compile(tree, "<safe>", "eval"))

print("-2 ** 2  =", safe_eval2("-2 ** 2"))     # -4 —— 先算幂再取负
print("(-2) ** 2 =", safe_eval2("(-2) ** 2")) # 4 —— 括号改变结合顺序

default:foo | 第二次: default:foo | 已缓存: True
swap: (2, 1)
加载类: <class 'json.decoder.JSONDecoder'>
拦截: 模块不在白名单: sys
按 key 取类: True
-2 ** 2  = -4
(-2) ** 2 = 4


---

## 结语

**Python 的动态特性让你能在运行时"改写规则"——它既是框架和库的基石，也是 bug 和安全问题的来源。用得好是魔法，用不好是灾难。**

> 本 Notebook 所有代码单元均真实执行，输出已嵌入；涉及不确定输出的示例（`id` / 字符串 `hash` /
> 插件枚举顺序 / 内存地址 repr）均已改造为确定性等价验证，详见各处「修正说明」。